# Decoding LLMs: Model Literacy for Social Scientists

<div class="alert alert-success"> 
    
### Learning Objectives
    
* **Interpret common model specifications** (Base vs. Instruct, Quantized vs. Full Precision, MoE vs. Dense) and understand their practical implications.
* **Evaluate tradeoffs** between capability, latency, context length, and cost when selecting a model for a specific research task.
* **Critically assess benchmark scores** and recognize when generic evaluations fail to capture social science workflows.
* **Apply quick, task-relevant checks** to compare model outputs and make informed decisions about which model fits your use case.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
🥊 **Challenge**: Interactive exercise. We'll work through these in the workshop!<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
📝 **Poll:** A Zoom poll to help you learn!<br>
🎬 **Demo**: Showing off something more advanced – so you know what Python can be used for!<br> 

### Sections
1. [Welcome to the Model Jungle](#welcome)
2. [Motivating Scenario: The Reddit Policy Scrape](#scenario)
3. [Model Taxonomy: What kind of model is this?](#taxonomy)
4. [Specs & Tradeoffs: Under the Hood](#specs)
5. [Trust Issues: Benchmarks vs. Reality](#benchmarks)
6. [The Vibe Check: Building an Eval Loop](#vibe-check)

<a id='welcome'></a>
## 1. Welcome to the Model Jungle

Interacting with a Large Language Model (LLM) through ChatGPT is straightforward. But when researchers decide to scale up their workflows—perhaps to process 10,000 interview transcripts or code thousands of survey responses—they quickly hit a wall of confusing jargon. 

When you look at a list of available models, you don't just see "AI." You see things like:
* `meta-llama/Llama-3-8B-Instruct`
* `mistralai/Mixtral-8x7B-v0.1`
* `google/gemma-2b-it-q4`

🔔 **Question:** If you have a strict budget, which one do you pick? Does a "70B" model perform 10x better than an "8B" model? Will a "Base" model follow your strict formatting instructions?

Today, we are going to demystify this jargon. We won't be doing heavy math; instead, we will use conceptual analogies and targeted examples to understand exactly what you are paying for—and what you are sacrificing—when you select a model from a dropdown menu.

<a id='ecosystem'></a>
## 2. The Ecosystem: Repositories vs. Interfaces

Before we can pick a model, we need to understand where they come from and how we access them. You will frequently hear names like Hugging Face, ChatGPT, and OpenRouter. How do they relate?

### Hugging Face: The "GitHub" of Machine Learning
Hugging Face is primarily a **repository**. It is where researchers (like Meta, Google, or university labs) upload the actual, multi-gigabyte files that make up an AI model (the "weights"). 
* **Analogy:** Hugging Face is like a massive public library. The models are the books. You can browse them, read their descriptions (Model Cards), and download them to your own computer if you have the massive hardware required to run them. 

### Interfaces: How We Actually Talk to the Models
Since social scientists usually don't have supercomputers lying around, we use web interfaces and APIs to let *someone else's* computer run the model.
* **Proprietary UIs:** You are already familiar with interfaces like ChatGPT or Claude. These are closed ecosystems; you can only talk to OpenAI or Anthropic models there.
* **Aggregators (OpenRouter, HuggingChat):** What if you want to test 50 different open-source models without setting up 50 accounts? This is where aggregator platforms come in. **OpenRouter** is a cool platform that provides a unified interface. It routes your prompt to whoever is hosting a specific model cheapest and fastest.

<div class="alert alert-success">  
💡 <i>Why OpenRouter today?</i> We are using OpenRouter for this workshop because it has a really easy, free Chat UI tier. It lets us play with dozens of models (like Meta's Llama or Mistral) in one place without needing a credit card or writing a line of code!
</div>

<a id='base-instruct'></a>
## 3. Model Taxonomy: Base vs. Instruct

Let's look at the first major distinction you will see in model names: **Base** vs. **Instruct** (often labeled as `-it` or `-chat`).

### The Base Model: A Pure Autocomplete Engine
A "Base" model has been trained on trillions of words from the internet with one simple goal: **Predict the next word.** It does not know it is an AI. It does not know it is supposed to answer questions. 

**Example:** Imagine you want to extract a location from a sentence. You send this prompt to a Base model:
> **Prompt:** "Extract the city from this text: I went to visit the Eiffel Tower in Paris."

> **Base Model Output:** "Extract the city from this text: I went to see Big Ben in London. Extract the city from this text: I went to see the Colosseum in Rome."

⚠️ **Warning:** The Base model didn't answer the prompt! It just noticed a pattern (a list of grammar exercises) and decided to write more of them. It is essentially a highly advanced smartphone autocorrect. 

### The Instruct (Chat) Model: The Helpful Assistant
An **Instruct** (or Chat) model is a Base model that has gone through a second phase of training called *Instruction Fine-Tuning*. Humans have shown it thousands of examples of prompts and the exact desired answers. It has been trained to stop autocompleting the internet and start *obeying commands*.

If we send the exact same prompt to the `-Instruct` version of that model, we get what we actually want:
> **Prompt:** "Extract the city from this text: I went to visit the Eiffel Tower in Paris."

> **Instruct Model Output:** "Paris"

<br>

### 🔔 Question: If Base models just autocomplete, why do we even use them?

If Instruct models are so much more helpful, why do companies release Base models, and why do researchers download them from Hugging Face? There are three main reasons you might actively *choose* a Base model over an Instruct model:

**1. The "Blank Slate" for Fine-Tuning**
Instruct models have been heavily trained to sound like a polite, conversational AI assistant. If you want to train a model to generate text in a highly specific style—like mimicking 19th-century literature or speaking in dense legal jargon—it is very hard to "un-train" the helpful AI voice of an Instruct model. A Base model is a blank slate, making it the perfect foundation to train your own custom model.

**2. Pure Autocomplete Workflows**
Sometimes, you actually *want* pure autocomplete. If you are using an AI coding assistant (like GitHub Copilot) or a creative writing tool, you just want the AI to seamlessly type the next sentence. If you used an Instruct model, it might abruptly stop your story to say, *"Sure, I can help you write the next paragraph! Here it is:"* **3. Strict Formatting (The "Chatty" Problem)**
Instruct models are designed to be conversational. If you ask an Instruct model to extract data into a strict JSON format, it will often break your code by adding conversational filler:
> *"Here is the data you requested!"*
> `[{"sentiment": "positive"}]`
> *"Let me know if you need anything else!"*

Because a Base model is just a pattern-matching engine, you can use **Few-Shot Prompting** (giving it 3 or 4 examples) to force it into a strict format without the conversational fluff. It will just continue the pattern perfectly.

<div class="alert alert-success">  
💡 <i>Where do they live?</i> Because everyday consumers find Base models confusing, companies hide them from consumer UIs (like ChatGPT). You will usually only find Base models in developer APIs, model repositories like Hugging Face, or advanced playgrounds like OpenRouter. 
</div>

<a id="specs-size"></a>

## 4.3 Model Size: What “1B” and “70B” Mean in Practice

After selecting an instruction-tuned model, one of the most visible specifications you will encounter is **model size**, typically reported in billions of parameters (e.g., `1B`, `8B`, `70B`).

Model size is a primary driver of:

- Computational cost  
- Inference latency  
- Reasoning capacity  
- Robustness to ambiguous inputs  

Understanding how size relates to performance is essential for making informed model choices.

---

### What Is a Parameter?

A **parameter** is a learned numerical value inside a neural network that influences how strongly different inputs affect the model’s outputs.

During training, the model adjusts these values to minimize prediction error on large text corpora.

In practice:

- More parameters → greater representational capacity  
- Fewer parameters → faster and cheaper inference  

However, more parameters do **not** guarantee better performance on all tasks.

⚠️ **Warning:** Parameter count is an imperfect proxy for quality. Training data, architecture, and fine-tuning often matter as much as size.

---

## Size Categories and Their Tradeoffs

For practical purposes, we can group modern LLMs into three broad size ranges.

These categories are approximate and overlap in real deployments.

---

### Ultra-Small Models (≈ 0.5B – 3B Parameters)

**Examples:** `Llama-3.2-1B`, `Qwen-0.5B`, `SmolLM`

**Typical Characteristics**

- Very low latency
- Low memory requirements
- Can run on consumer hardware
- Limited reasoning depth

**Advantages**

- Extremely inexpensive at scale
- Suitable for offline or edge deployment
- Predictable performance on narrow tasks

**Limitations**

- Weak multi-step reasoning
- Sensitive to prompt phrasing
- Poor performance on open-ended tasks

**Common Use Cases**

- Intent classification
- Topic labeling
- Spam and moderation filters
- Prompt routing
- Lightweight extraction pipelines

💡 **Tip:** In large systems, ultra-small models are often used as “infrastructure models” that route or filter inputs before heavier processing.

---

### Small / Medium Models (≈ 7B – 14B Parameters)

**Examples:** `Llama-3-8B`, `Mistral-7B`, `Gemma-2-9B`

**Typical Characteristics**

- Good balance of speed and capability
- Moderate hardware requirements
- Strong general-purpose performance

**Advantages**

- Affordable via API
- Reasonable reasoning ability
- Reliable for many applied tasks

**Limitations**

- Struggles with very complex logic
- Less robust to long reasoning chains
- Limited tolerance for noisy inputs

**Common Use Cases**

- Document summarization
- Information extraction
- RAG-based systems
- Survey response analysis
- General research assistants

These models are frequently the **default choice** in applied research workflows.

---

### Large / Frontier Models (≈ 70B+ Parameters)

**Examples:** `Llama-3-70B`, flagship GPT and Claude models

**Typical Characteristics**

- High reasoning capacity
- Strong handling of ambiguity
- Better generalization

**Advantages**

- More reliable on complex tasks
- Better at multi-step reasoning
- Strong performance on creative and analytical tasks

**Limitations**

- High monetary cost
- Higher latency
- Significant infrastructure demands

**Common Use Cases**

- Complex qualitative analysis
- Long-form reasoning
- Code generation
- Evaluation of other models
- High-stakes decision support

⚠️ **Warning:** Large models should not be treated as “default” options. Their benefits are concentrated in complex or high-uncertainty tasks.



## 🥊 Challenge 1: Medium vs. Frontier Models in Practice (5–10 Minutes)

In this exercise, you will compare how two commonly available models interpret the same social research text.

You will use:

- One medium-sized model (≈9B parameters)
- One large / frontier model (≈300B+ parameters)

The goal is to observe what additional analytical capacity you gain when using a much larger model.

---

### Learning Goal

By the end of this exercise, you should be able to:

- Identify concrete differences in interpretation between mid-range and flagship models
- Evaluate whether those differences matter for your research context
- Connect output quality to cost and latency

---

## Step 1: Select Two Models

Using the provided interface, select:

1. One medium model (≈9B)
2. One frontier model (≈300B+)

Record their names:

- Medium model: [Nemotron Nano 9B V2 (free) ](https://openrouter.ai/nvidia/nemotron-nano-9b-v2:free)
- Frontier model: [Qwen: Qwen3 VL 235B A22B Thinking](https://openrouter.ai/qwen/qwen3-vl-235b-a22b-thinking)


⚠️ **Warning:** Use the same two models throughout this exercise.

---

## Step 2: Use the Standardized Prompt

Copy the prompt and text below exactly. Do not modify them.

---

**Prompt**

You are analyzing social research text.

Summarize the passage and identify two major themes.  
For each theme, provide one supporting quotation.

**Text**

Last year, when the city announced the new housing assistance program, I was hopeful at first.

The application website kept crashing, and the phone line was always busy. After three weeks, I finally reached someone who told me my documents were “under review.” No one could explain what that meant.

My neighbor got approved in two days. She told me it helped that her caseworker “knew her situation already.” I’m not sure what that means either.

By the time I received a response, winter had already started. They said I was missing one form, even though I had submitted it twice. When I tried to appeal, I was told the deadline had passed.

I’m grateful that some people get help. I just wish the process didn’t make you feel like you were doing something wrong by asking.

---

## Step 3: Run Both Models

Submit the same prompt to:

1. The medium model  
2. The frontier model  

---

## Step 4: Discussion

- Where did the frontier model improve meaningfully?
- Where were outputs similar?
- Did the larger model notice anything the smaller one missed?
- Were the differences worth the added cost?

---

## 🔔 Reflection Question

Consider your own research workflow.

> Which parts of your work would benefit most from the frontier model’s output, and which could be handled by a medium model?

Write one concrete example for each.

---

## Interpreting Your Results

When comparing outputs, focus on patterns rather than isolated errors.

Look for:

- Systematic simplification in the medium model
- Increased nuance in the frontier model
- Differences in evidence selection
- Stability of interpretations

⚠️ **Warning:** A single strong response does not indicate reliability. Look for consistency across tasks.

---

## Summary

This exercise illustrates how model size influences:

- Interpretive depth
- Sensitivity to social context
- Handling of ambiguity
- Cost–performance tradeoffs

These differences become more consequential as analyses scale.


<a id="specs-architecture"></a>
## 4.5 Model Architecture: Dense vs. Mixture-of-Experts (MoE)

### Why This Matters
Two models with similar parameter counts can behave very differently depending on how they are built. 

Architecture affects:
- **Cost** (via API or local electricity)
- **Latency** (Time to first token)
- **Stability** (Consistency of reasoning)
- **Scaling behavior**

---

### Dense Models (The Standard)
In a **dense model**, all parameters are used for every token. Every forward pass uses the entire network.

* **Analogy:** A corporate committee where every single member—from the accountant to the graphic designer—must review, analyze, and approve every single email that comes into the company inbox.
* **Properties:**
  * Simple architecture
  * Predictable performance
  * High compute cost
  * Easier to fine-tune

*(Most small and mid-sized models, like `Llama-3-8B`, are dense.)*

---

### Mixture-of-Experts (MoE) Models (The Triage System)



In a **Mixture-of-Experts (MoE)** model, only part of the network is active per token. A mathematical routing mechanism selects which specialized “experts” process the input.

* **Analogy:** A hospital triage system. If a patient comes in with a broken bone, the triage nurse routes them specifically to the orthopedist, not the cardiologist or the neurologist.
* **Properties:**
  * Very large total parameter count
  * Lower per-token compute cost
  * More complex training
  * Potentially less stable on highly ambiguous prompts

---

<div class="alert alert-warning">
⚠️ <b>Parameter Count Can Be Misleading</b><br>

An MoE model (like `Mixtral-8x7B`) may be advertised as having “~47B parameters.” However, only a fraction (roughly 14B) may be active per token. Do not interpret parameter count alone when judging the speed or cost of a model!
</div>

---

### 🥊 Challenge 1 (5 minutes): Identify Architecture
Using the OpenRouter or Hugging Face search bar:
1. Find one dense model.
2. Find one MoE model.
3. Note down:
   - Parameter count
   - Architecture type
   - Estimated cost per 1M tokens (if available)

💡 **Tip:** Model names containing **"Mix"**, **"MoE"**, **"Experts"**, **"<Experts>x<ParamSize>"** are almost always mixture-of-experts models.

***



<a id="specs-quantization"></a>
## 4.6 Quantized Models: Running LLMs on Limited Hardware

### Why This Matters
So far, you have interacted with hosted models. But running models locally (on your own laptop or a university server) introduces strict hardware constraints:
- Memory (VRAM)
- Storage
- Compute power

**Quantization** is the mathematical trick that makes local inference feasible for researchers without supercomputers.

---

### What Is Quantization?



Quantization reduces the numerical precision of model weights.

During training, models typically use high-precision 32-bit floating-point values. Quantization rounds these numbers down so the model uses:
- 16-bit
- 8-bit (`INT8`)
- 4-bit (`INT4` or `q4`)

* **The Analogy:** Think of quantization like compressing a massive, high-resolution RAW photograph into a smaller JPEG. You lose some fine-grained pixel detail, but the file size shrinks dramatically while the core image remains highly recognizable.

---

### Tradeoffs

**Benefits:**
- Dramatically smaller file sizes
- Lower memory use (VRAM)
- Faster loading and inference on consumer hardware

**Costs:**
- Reduced numerical precision
- Possible quality degradation in complex reasoning or heavy math
- Limits on your ability to fine-tune the model further

---

### 🔔 Question
If an 8-bit quantized model and a full-precision (32-bit) model both score similarly on a basic sentiment analysis benchmark, why might a researcher still prefer to pay extra for the full-precision model?

💡 **Tip:** When browsing models, look for common quantization indicators in the filename or tags, such as: `int8`, `4bit`, `GPTQ`, `AWQ`, or `GGUF`.

***

<a id="hf-bridge"></a>
## 5. Bridge: From Hosted APIs to Downloaded Models

### Why This Section Exists
We are now moving from:
> *Using deployed models via APIs (like OpenRouter)* > to  
> *Downloading and running models yourself*

This changes how you must think about your workflow. You are now responsible for:
- Reproducibility
- Infrastructure limits
- Reading the documentation carefully

---

### Hugging Face as a Model Repository



Hugging Face is the standard repository for open-weight models. It provides:
- The actual model weights (gigabytes of files)
- Training descriptions
- Evaluation notes
- Example Python code

Every reputable model has a public **Model Card** (a `README.md` file attached to the repository). Learning to read these cards is a fundamental part of model literacy.

---

### 🎬 Demo: Exploring a Model Page
*Instructor will share their screen and navigate to a popular Hugging Face model repository.*

We will examine:
1. Architecture & Parameter count
2. Quantization options
3. Licensing (Can you use this for commercial research?)
4. Example usage code

---

### 🥊 Challenge 2 (7 minutes): Read a Model Card
1. Open a Hugging Face model page (e.g., search for `meta-llama/Meta-Llama-3-8B-Instruct`).
2. Locate and read the following sections:
   - Training data description
   - Intended use cases
   - Known Limitations
   - Hardware requirements

---

<div class="alert alert-warning">
⚠️ <b>Silence is Meaningful</b><br>

Many models have incomplete documentation. If a model card is missing a "Limitations" section or refuses to disclose its "Training Data," that missing information is itself a massive red flag for rigorous academic research.
</div>